<a href="https://colab.research.google.com/github/MaggieHDez/ClassFiles/blob/IDA/data_profiling_255879.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Actividad: Data Profiling con PySpark
>Clase: Ingeniería de Datos Avanzada\
>Alumno: Margarita Cristina Hernández Delgadillo\
>Matrícula: 255879\
>Fecha: 03 de Mayo de 2026

In [15]:
!python -m pip install "pymongo[srv]"
!pip install pyspark

##1. Verificación de versión de PySpark

In [16]:
# Verificamos si PySpark ya está instalado en el entorno
try:
    import pyspark
    print("PySpark ya está instalado")
    print("Versión:", pyspark.__version__)
except ModuleNotFoundError:
    # Si no está instalado, lo instalamos
    print("PySpark no está instalado. Instalando...")
    !pip install pyspark -q
    import pyspark
    print("Instalación completada")
    print("Versión:", pyspark.__version__)

PySpark ya está instalado
Versión: 4.0.2


##2. Carga del archivo con PySpark

In [17]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("NYC Taxi Profiling") \
.getOrCreate()

!wget -O taxi.csv "https://www.dropbox.com/scl/fi/ya6wwi1ouvu7b5ng00zu3/yellow_tripdata_2016-03.csv?rlkey=49gbpo35mmh7p2codjw4kcfd3&dl=1"

df = spark.read.csv("taxi.csv", header=True, inferSchema=True)

--2026-05-04 04:01:56--  https://www.dropbox.com/scl/fi/ya6wwi1ouvu7b5ng00zu3/yellow_tripdata_2016-03.csv?rlkey=49gbpo35mmh7p2codjw4kcfd3&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.6.18, 2620:100:601c:18::a27d:612
Connecting to www.dropbox.com (www.dropbox.com)|162.125.6.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc49c3d3776749ba2d1cf5fa8c38.dl.dropboxusercontent.com/cd/0/inline/C_z-T6dhaly9FvBpTE99H6njANDpB997N7-Y6P1fq_tHc3yOLVP0BElZHhbgTsLc0y4YSLlQH2tVh8BxRNi5mqZDw1SqkjrT5QeE0UxHML6SXyCf7zMjf4lX0scpoZ2xEm4/file?dl=1# [following]
--2026-05-04 04:01:57--  https://uc49c3d3776749ba2d1cf5fa8c38.dl.dropboxusercontent.com/cd/0/inline/C_z-T6dhaly9FvBpTE99H6njANDpB997N7-Y6P1fq_tHc3yOLVP0BElZHhbgTsLc0y4YSLlQH2tVh8BxRNi5mqZDw1SqkjrT5QeE0UxHML6SXyCf7zMjf4lX0scpoZ2xEm4/file?dl=1
Resolving uc49c3d3776749ba2d1cf5fa8c38.dl.dropboxusercontent.com (uc49c3d3776749ba2d1cf5fa8c38.dl.dropboxusercontent.com)... 162.125.6.15, 2620:100:601c:15::

##3. Data Profiling con PySpark

###3.1 Estructura del dataset (schema, filas y columnas)

In [18]:
# Revisión de estructura del dataset

# Schema del DataFrame
print("Schema del DataFrame:")
df.printSchema()

# Número de filas y columnas
# Observaciones
num_filas, num_columnas = df.count(), len(df.columns)

print(f"Número de filas: {num_filas}")
print(f"Número de columnas: {num_columnas}")
print(f"Columnas: {df.columns}")

Schema del DataFrame:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)

Número de filas: 12210952
Número de columnas: 19
Columnas: ['VendorID', 'tpep_pickup_datetime',

###3.2 Estadísticos descriptivos

In [19]:
# Estadísticos descriptivos de todas las columnas numéricas
print("Estadísticos descriptivos de todas las columnas numéricas:\n")
df.describe().show(truncate=False)

Estadísticos descriptivos de todas las columnas numéricas:

+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------+-----------------+
|summary|VendorID          |passenger_count   |trip_distance     |pickup_longitude   |pickup_latitude   |RatecodeID        |store_and_fwd_flag|dropoff_longitude  |dropoff_latitude  |payment_type      |fare_amount       |extra              |mta_tax            |tip_amount        |tolls_amount       |improvement_surcharge|total_amount     |
+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+

###3.3 Valores nulos

In [20]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType

# Separar columnas numéricas y no numéricas
tipos_numericos = (DoubleType, FloatType, IntegerType, LongType)

nulos = df.select([
    spark_sum(
        when(
            col(c).isNull() | isnan(col(c)), 1  # isnan solo para numéricos
        ).otherwise(0)
        if isinstance(df.schema[c].dataType, tipos_numericos)
        else
        when(col(c).isNull(), 1).otherwise(0)  # solo isNull para no numéricos
    ).alias(c)
    for c in df.columns
])

print("Valores nulos por columna:")
nulos.show(truncate=False)

Valores nulos por columna:
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RatecodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|0       |0                   |0                    |0              |0            |0               |0              |0         |0     

###3.4 Análisis de correlación entre variables numéricas

In [21]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd

# Seleccionar solo columnas numéricas
columnas_numericas = [f.name for f in df.schema.fields
                      if str(f.dataType) in ['DoubleType()', 'IntegerType()',
                                              'LongType()', 'FloatType()']]

print("Columnas numéricas:", columnas_numericas)

# Ensamblar columnas en un vector
assembler = VectorAssembler(inputCols=columnas_numericas, outputCol="features",
                             handleInvalid="skip")
df_vector = assembler.transform(df).select("features")

# Calcular matriz de correlación (método Pearson)
matrix = Correlation.corr(df_vector, "features")

# Convertir a un formato legible
corr_matrix = matrix.collect()[0][0].toArray()
corr_df = pd.DataFrame(corr_matrix, index=columnas_numericas, columns=columnas_numericas)

print("\nMatriz de Correlación (Pearson):")
print(corr_df.round(2))

Columnas numéricas: ['VendorID', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']

Matriz de Correlación (Pearson):
                       VendorID  passenger_count  trip_distance  \
VendorID                   1.00             0.29           -0.0   
passenger_count            0.29             1.00           -0.0   
trip_distance             -0.00            -0.00            1.0   
pickup_longitude          -0.05            -0.02           -0.0   
pickup_latitude            0.05             0.02            0.0   
RatecodeID                -0.00            -0.01            0.0   
dropoff_longitude         -0.05            -0.02           -0.0   
dropoff_latitude           0.05             0.02            0.0   
payment_type              -0.02             0.01            0.0   
fare_amount

##4. Profiling con ydata-profiling

###4.1 Instalación ydata-profiling

In [22]:
!pip install ydata-profiling[pyspark] -q # -q para que la instalación sea silenciosa

###4.2 Generación del reporte

In [23]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

# Se usa una muestra del 10% (~1.2M filas) porque el dataset completo
# (~12M filas) provoca OutOfMemoryError al calcular la correlación de
# Spearman en Colab. La muestra sigue siendo estadísticamente representativa.
df_muestra = df.sample(fraction=0.1, seed=42)

# Se convierten las columnas TIMESTAMP a string porque ydata_profiling
# tiene un bug conocido al procesar fechas provenientes de Spark,
# lanzando KeyError: 'n_invalid_dates' al generar el reporte.
columnas_timestamp = [
    f.name for f in df_muestra.schema.fields
    if isinstance(f.dataType, TimestampType)
]

df_muestra_fix = df_muestra
for c in columnas_timestamp:
    df_muestra_fix = df_muestra_fix.withColumn(c, col(c).cast("string"))

# Se genera el reporte con la muestra corregida.
# El parámetro explorative=True activa análisis más profundos
# y acelera la generación del reporte.
from ydata_profiling import ProfileReport

profile = ProfileReport(df_muestra_fix, explorative=True)

# Se muestra el reporte directamente dentro del notebook de Colab
# usando un iframe embebido, sin necesidad de abrir un archivo externo.
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]